# H SC 790 · Lab 6 · Build a Detector and Measure How Often It Cries Wolf

**Week 7.** Due Thursday, March 11, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Both classifiers scored; three detectors compared
- One artifact and one real signal explained
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

A year (2025) of emergency department visits from the five hospitals connected to syndromic surveillance. Classify chief complaints into syndromes, build daily series, run C1, C2, and a CUSUM, and score detection against the year's known events.

Keep some events out of your threshold tuning, so you can score on events you did not tune on.

In [ ]:
import re
import matplotlib.pyplot as plt

ed = pd.read_csv(COURSE_URL + "lab06/ed_visits_2025.csv.gz", dtype={"zip": str}, parse_dates=["visit_ts"])
labeled = pd.read_csv(COURSE_URL + "lab06/labeled_sample.csv", parse_dates=["visit_ts"])
events = pd.read_csv(COURSE_URL + "lab06/outbreaks_2025.csv", parse_dates=["start_date", "end_date"])
hospitals = pd.read_csv(COURSE_URL + "lab06/hospitals.csv")
ed["day"] = ed.visit_ts.dt.normalize()
print(len(ed), "visits")
events

## 1. Explore the complaints

Sample chief complaints by hospital. List the abbreviations, misspellings, and negations you find. Do all hospitals record complaints the same way, all year?

In [ ]:
# YOUR CODE HERE

## 2. Build two classifiers

A rule-based classifier for three syndromes (`ili`, `gi`, `heat`) and a learned one. Score both against the hand-labeled sample: sensitivity and predictive value, overall and by hospital. The ILI and GI rules are the slide's.

In [ ]:
# --- rules: transparent, no training data (slide) ---
NEG = r'\b(denies|no|without|neg)\s+[\w ]{0,25}'
def is_ili(cc):
    t = re.sub(NEG, ' ', cc.lower())
    fever = re.search(r'fev|febrile|temp', t)
    resp  = re.search(r'cough|sore thr|st\b', t)
    return bool(fever and resp)
def is_gi(cc):
    t = re.sub(NEG, ' ', cc.lower())
    return bool(re.search(r'vomit|diarr|n/v|nause', t))
def is_heat(cc):
    # YOUR CODE HERE
    pass

def sens_ppv(pred, truth):
    """pred and truth are boolean Series."""
    tp = (pred & truth).sum()
    return pd.Series({"n_pos": truth.sum(), "sens": tp / max(1, truth.sum()), "ppv": tp / max(1, pred.sum())}).round(3)

In [ ]:
# YOUR CODE HERE

One hospital stands out. Split its labeled visits by date and score again to find out when and why.

In [ ]:
# YOUR CODE HERE

## 3. Make the time series

Daily counts by syndrome: for the county, by hospital, and by ZIP code. Plot them before running anything.

In [ ]:
# YOUR CODE HERE

## 4. Implement C1, C2, and a CUSUM

With a floor on the standard deviation and a minimum count. C2 and the CUSUM are the slide's; write C1 (the same baseline with no gap). Mark the alerts on a plot.

In [ ]:
# --- EARS C2: 7-day baseline, 2-day gap (slide) ---
def c2(counts, thresh=3.0, sd_floor=1.0):
    s  = pd.Series(counts, dtype=float)
    b  = s.shift(3).rolling(7)       # days t-9 .. t-3
    z  = (s - b.mean()) / b.std().clip(lower=sd_floor)
    return z, (z > thresh) & (s >= 5)  # minimum count

# --- one-sided CUSUM on the same z-scores (slide) ---
def cusum(z, k=0.5, h=4.0):
    S, out = 0.0, []
    for v in z.fillna(0):
        S = max(0.0, S + v - k)
        out.append(S)
    S = pd.Series(out, index=z.index)
    return S, S > h

def c1(counts, thresh=3.0, sd_floor=1.0):
    # YOUR CODE HERE
    pass

In [ ]:
# YOUR CODE HERE

## 5. Score detection

Against the event file: sensitivity (share of events with at least one alert), false alerts per month (alert days outside any event of that syndrome), and days to first alert, for each detector at three thresholds. Monitor each local event's own ZIP codes as well as the county series.

In [ ]:
# YOUR CODE HERE

## 6. Explain two signals

Find one alert that is a real outbreak and one that is a data artifact. Say how you told them apart. Total visit volume, by hospital, is worth monitoring too.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. Which classifier would you deploy?** Give sensitivity and predictive value for each, by hospital.

*Your answer here.*

**2. How many false alerts per month?** For each detector at your chosen threshold, across all series you monitor.

*Your answer here.*

**3. What did stratifying reveal?** Show a signal that county totals hid, and one that was a data artifact.

*Your answer here.*

**4. Who gets the alert, and what do they do?** Name the role, the first three steps, and how long each should take.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab06/` of your course repository.